[Course Overview](<00-course-overview.ipynb>)

# 01 - What Is a Database: tables, rows, and why there is more than one

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/01-what-is-a-database.ipynb)

Every SQL tutorial assumes you already know what a database is, and this notebook starts one step earlier. There's no SQL for you to learn here yet: what you get is the picture the rest of the course builds on, which is what a table is, how it differs from the spreadsheets you've probably used, why a database splits its facts across several tables, and what SQL is for. Every query you write later is a question asked of that picture, and a question is much easier to ask well when you can see what you're asking about.

*New here? The [Course Overview](<00-course-overview.ipynb>) explains how to run these notebooks. Viewing this on GitHub shows a preview with saved results; it won't execute anything.*

> **By the end of this notebook you'll be able to:** say what one row and one column represent in any table, explain how a table differs from a spreadsheet, show why a database stores each fact once and reconnects tables through a shared column, and describe what SQL is for before you've written a line of it.

> **Read the full lesson:** this notebook is the hands-on companion to [Lesson 1 of the All About Data & More curriculum](LESSON_1_URL), which walks through the reasoning in more depth.

## 0. How this notebook works

A notebook is a document made of **cells**. Text cells, like this one, explain things. Code cells, the grey boxes, hold something to run, and when you run one its result appears directly underneath it.

- **To run a cell**, click it and press **Shift + Enter**, or click the ▶ button at its left edge. The notebook moves on to the next cell, so you can keep pressing Shift + Enter to work down the page.
- **Run the cells in order, from the top.** Later cells rely on what earlier ones set up, so if something says a table doesn't exist, go back up and run the cells you skipped.
- **You don't need to read the code yet.** Every code cell in this notebook is already written for you. Run it and look at the result underneath; the code itself becomes something you write from `02 - SQL Basics` on.

Run the cell below first. It loads the small helper that lets the other cells talk to a database; any cell that starts with `%%sql` then runs against a small practice database that lives inside this notebook, so there's nothing to install.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


## 1. Start with a small shop

Say you run a small shop with four regular customers, Ada, Bola, Chidi, and Dami, and between them they've placed five orders. In this course, **data** means nothing grander than that: facts about real things, like who your customers are and what they've ordered, written down in an organized way so you can find them again.

The cell below writes those facts into two tables. These customers and orders are invented, and they're the same rows the later lessons use from `04 - JOINs` onwards, so they're worth getting to know.

In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


Now look at the customers:

In [3]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers;

customer_id,customer_name,credit_limit
1,Ada,100
2,Bola,200
3,Chidi,300
4,Dami,200


## 2. A table is a stricter spreadsheet

If you've used Excel or Google Sheets, most of that is familiar. Each **row** is one customer, and each **column** is one fact about every customer: an ID number, a name, and a credit limit, which here means how much that customer is allowed to owe the shop at once. The column names sit at the top and say what every value underneath them means.

What makes it a table rather than a spreadsheet is that it's strict. A spreadsheet lets you type anything into any cell, a number in one, a note to yourself in the next, two cells merged because it looked tidier. A table doesn't. Every row has the same columns, and every column holds one kind of value, so `credit_limit` holds whole numbers for every customer and never, in one row, the words "ask Bola". That strictness feels fussy at first, but it's what lets a computer work with the table without a person checking each cell, since adding up a column only makes sense if every value in it is a number.

> **Worth knowing:** SQLite, the database engine behind these practice cells, is more relaxed about column types than most engines unless a table is declared [`STRICT`](https://www.sqlite.org/stricttables.html), while SQL Server and PostgreSQL refuse a value of the wrong type outright. The habit is the same either way: decide what kind of value each column holds, and keep it that way.

## 3. Two things a table does differently

### Rows don't have positions

In a spreadsheet, Bola is on row 3, and you can point at row 3. A table's rows have no built-in position. The [SQLite documentation](https://www.sqlite.org/lang_select.html) says that unless a query asks for a particular order, "the order in which the rows are returned is undefined", which means the same table can hand back its rows in a different order from one day to the next.

So you can't identify Bola by where she sits. You identify her by her values, and that's what `customer_id` is for: a number that no two customers share, so `customer_id = 2` always means Bola, wherever her row happens to come out. A column like that is called a **primary key**. Here's Bola, found by her ID rather than her position:

In [4]:
%%sql
SELECT
    customer_id,
    customer_name,
    credit_limit
FROM customers
WHERE customer_id = 2;

customer_id,customer_name,credit_limit
2,Bola,200


### An empty value means unknown

A blank spreadsheet cell can mean zero, not yet, or someone forgot, and you have to guess which. A table has a specific marker for a value that's missing, `NULL`, and it means **unknown**, not zero and not empty text. That distinction sounds small, and `02 - SQL Basics` shows it changing the answer to a simple question.

## 4. Why a database has more than one table

Here are the orders:

In [5]:
%%sql
SELECT
    order_id,
    customer_id,
    amount
FROM orders;

order_id,customer_id,amount
101,1,40
102,1,60
103,2,80
104,2,90
105,3,140


Each row is one order, and notice what isn't in it: no customer names, no credit limits, only a `customer_id`. That number points back to the customers table, so order 101 belongs to customer 1, who is Ada. Ada has two orders, Bola has two, Chidi has one, and Dami has none, so she appears in the customers table and nowhere in this one.

<p align="center">
  <img src="graphics/01_tables_connected.png" width="460" alt="Customers 1 to 3 each connect to their orders through customer_id; customer 4, Dami, connects to no order">
</p>

A **database** is a collection of tables like these, stored together and connected through shared columns like `customer_id`, so a computer can find a customer, find every order that points to them, and put the two back together whenever someone asks.

## 5. Common mistake: putting everything in one table

The obvious alternative is one big table with everything on each row. Run this to build it:

In [6]:
%%sql
CREATE TABLE orders_flat (
    order_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    credit_limit INTEGER,
    amount INTEGER
);
INSERT INTO orders_flat VALUES
    (101, 'Ada', 100, 40),
    (102, 'Ada', 100, 60),
    (103, 'Bola', 200, 80),
    (104, 'Bola', 200, 90),
    (105, 'Chidi', 300, 140);
SELECT
    order_id,
    customer_name,
    credit_limit,
    amount
FROM orders_flat;

5 rows inserted


order_id,customer_name,credit_limit,amount
101,Ada,100,40
102,Ada,100,60
103,Bola,200,80
104,Bola,200,90
105,Chidi,300,140


It looks simpler, and for reading it is. The trouble starts when something changes. Say Ada's credit limit goes up to 150, and whoever makes the change updates order 101 and misses order 102. Run it and see what the table says afterwards:

In [7]:
%%sql
UPDATE orders_flat
SET credit_limit = 150
WHERE order_id = 101;
SELECT
    order_id,
    customer_name,
    credit_limit,
    amount
FROM orders_flat
WHERE customer_name = 'Ada';

1 row updated


order_id,customer_name,credit_limit,amount
101,Ada,150,40
102,Ada,100,60


So what is Ada's credit limit, 150 or 100? The table now disagrees with itself, and nothing in it says which value is right. Dami has a worse problem, since she has no orders and so has no row at all; a table of orders has nowhere to keep a customer who hasn't ordered yet.

Now make the same change in the two-table design. Ada's limit lives on exactly one row of `customers`, so there's only one row to change, and every order reaches the new value through `customer_id`:

In [8]:
%%sql
UPDATE customers
SET credit_limit = 150
WHERE customer_id = 1;
SELECT
    o.order_id,
    c.customer_name,
    c.credit_limit,
    o.amount
FROM orders AS o
INNER JOIN customers AS c
    ON o.customer_id = c.customer_id
WHERE c.customer_id = 1;

1 row updated


order_id,customer_name,credit_limit,amount
101,Ada,150,40
102,Ada,150,60


Both of Ada's orders now show 150, because there was only ever one place her limit was written.

<p align="center">
  <img src="graphics/01_one_table_vs_two.png" width="560" alt="In one big table, updating one of two rows leaves Ada with two different limits and Dami with no row; with customers and orders, one row changes and both orders see it">
</p>

Splitting the data into a customers table and an orders table means **each fact is stored once**. The cost is that answering a question about both, like "how much has each customer spent against their limit?", means connecting the tables again, and that connection, called a **join**, gets its own notebook in `04 - JOINs`. `02` and `03` deliberately start with a single simpler table that writes the customer's name on every order, because one table is easier to learn on, and `04` brings the two tables back.

## 6. What SQL is for

**SQL** stands for Structured Query Language, and it's the language you use to ask a database questions (and, later in the course, to change what's stored). The part worth understanding now is *how* you ask. SQLite's documentation calls SQL ["a declarative language, not a procedural language"](https://www.sqlite.org/queryplanner.html): "you tell the system what you want to compute, not how to compute it." You don't write instructions to open the orders table, walk through it row by row, and keep a running total; you describe the answer you want, and the database works out the steps.

Take a question in plain English: which customers have orders adding up to more than 100? You can answer it from the two tables with a pencil. You'd match each order to its customer through `customer_id`, add up the amounts per customer, and keep the totals over 100: Bola's orders come to 170 and Chidi's to 140, so they qualify, while Ada's come to exactly 100, which isn't *more* than 100, and Dami has no orders at all.

Here's that same question written in SQL. Don't try to decode it yet, just run it and check it agrees with your pencil:

In [9]:
%%sql
SELECT
    c.customer_name,
    SUM(o.amount) AS total_ordered
FROM customers AS c
INNER JOIN orders AS o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_name
HAVING SUM(o.amount) > 100
ORDER BY
    c.customer_name;

customer_name,total_ordered
Bola,170
Chidi,140


The steps you just did by hand, keeping some rows, adding up per customer, and connecting two tables, are close to the next few notebooks in order: filtering rows in `02`, adding up groups in `03`, and joining tables in `04`. By the end of `04` you'll be able to write that query yourself and explain every line of it.

SQL is shared across many database engines, like SQLite, PostgreSQL, MySQL, and SQL Server, and each one has its own **dialect**, so a few details differ from engine to engine. The practice cells in this course run SQLite, and each notebook ends by running the same ideas on SQL Server, pointing out the differences when they matter.

## 7. Try it yourself: read a sign-up sheet as tables

Here's a sign-up sheet from an invented gym. Run the cell to see it:

In [10]:
%%sql
CREATE TABLE signup_sheet (
    member_name TEXT,
    member_email TEXT,
    class_name TEXT,
    class_day TEXT
);
INSERT INTO signup_sheet VALUES
    ('Kemi', 'kemi@example.com', 'Spin', 'Monday'),
    ('Kemi', 'kemi@example.com', 'Yoga', 'Wednesday'),
    ('Tolu', 'tolu@example.com', 'Spin', 'Monday'),
    ('Musa', 'musa@example.com', 'Yoga', 'Wednesday');
SELECT
    member_name,
    member_email,
    class_name,
    class_day
FROM signup_sheet;

4 rows inserted


member_name,member_email,class_name,class_day
Kemi,kemi@example.com,Spin,Monday
Kemi,kemi@example.com,Yoga,Wednesday
Tolu,tolu@example.com,Spin,Monday
Musa,musa@example.com,Yoga,Wednesday


Before looking at the [solutions notebook](<solutions/01-what-is-a-database-solutions.ipynb>), answer two questions, in writing if you can:

1. **What does one row represent?** Be precise: it isn't "a member".
2. **Which facts are repeated, and what separate tables would you split them into?** For each table, say what one row of it represents and which columns it needs.

The solutions notebook builds your answer as real tables, so you can check it by running it.

## 8. Scale it up: a real company's database

Four customers and five orders make the idea easy to see, and real databases use exactly the same idea at a size you can't see all at once. **AdventureWorks2022** is Microsoft's sample database for an invented bike company, and it runs on **SQL Server**, a different database engine from the SQLite behind the practice cells. Cells that start with `%%tsql` run there, and every notebook in the course ends with a section like this one.

### Set up AdventureWorks

SQL Server is a separate program from this notebook, so it needs one setup step before the `%%tsql` cells can run:

- **In Google Colab**, the cell below installs SQL Server and AdventureWorks inside your Colab session for you. The first run takes a few minutes and prints each step as it goes. Colab gives you a fresh machine every session, so run it again whenever you come back to the notebook later.
- **On your own machine**, it checks that SQL Server is reachable and tells you what's missing if it isn't; the [Course Overview](<00-course-overview.ipynb>) covers installing SQL Server and AdventureWorks locally.

If you'd rather not set anything up, every `%%tsql` cell below keeps its saved result, so you can read the section without running it.

In [11]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


### How many tables does a real business need?

AdventureWorks groups its tables into **schemas**, which work like folders: `Sales` for customers and orders, `Production` for products, `Person` for people's names and addresses, and so on. This counts the tables in each one:

In [12]:
%%tsql
SELECT
    TABLE_SCHEMA AS schema_name,
    COUNT(*) AS tables
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_TYPE = 'BASE TABLE'
GROUP BY
    TABLE_SCHEMA
ORDER BY
    tables DESC;

schema_name,tables
Production,25
Sales,19
Person,13
HumanResources,6
Purchasing,5
dbo,3


71 tables, every one of them holding one kind of thing, the same way `customers` and `orders` do. You never need all of them at once; a question about sales usually touches two or three.

### The same pattern as the shop

Here are the first few orders. Like the shop's `orders` table, each row is one order, and it identifies the customer only by a number, `CustomerID`, with no name anywhere:

In [13]:
%%tsql
SELECT TOP 5
    SalesOrderID,
    CustomerID,
    OrderDate,
    TotalDue
FROM Sales.SalesOrderHeader
ORDER BY
    SalesOrderID;

SalesOrderID,CustomerID,OrderDate,TotalDue
43659,29825,2011-05-31 00:00:00,23153.2339
43660,29672,2011-05-31 00:00:00,1457.3288
43661,29734,2011-05-31 00:00:00,36865.8012
43662,29994,2011-05-31 00:00:00,32474.9324
43663,29565,2011-05-31 00:00:00,472.3108


Customer 29825 placed the very first order. Following that number into the customers table finds exactly one row, because `CustomerID` is that table's primary key:

In [14]:
%%tsql
SELECT
    CustomerID,
    PersonID,
    StoreID,
    TerritoryID
FROM Sales.Customer
WHERE CustomerID = 29825;

CustomerID,PersonID,StoreID,TerritoryID
29825,1045,1046,5


And notice that this row is made of more ID numbers: `PersonID` points into the `Person` schema for a name, `StoreID` to the shop this customer buys for, and `TerritoryID` to a sales region. A real database is mostly this, facts stored once and connected through shared columns. Going the other way, from the customer to every order that points at them:

In [15]:
%%tsql
SELECT COUNT(*) AS orders_placed
FROM Sales.SalesOrderHeader
WHERE CustomerID = 29825;

orders_placed
12


Twelve orders, each one reaching this customer's details through a single number, so a change to the customer is made once and all twelve see it. That's the shop's design, running a business with 19,820 customers and 31,465 orders.

## The habit to carry into the next notebook

Everything in this notebook comes down to one question you can ask of any table before you query it: **what does one row represent, and what does each column mean?** Asking it first is what stops a correct-looking query from answering the wrong thing, and every notebook after this one comes back to it.

## Go deeper

- **[CS50's Introduction to Databases with SQL](https://cs50.harvard.edu/sql/)**, Harvard. A free university course whose first lecture covers this same ground, tables, keys, and why data is split, with videos and problem sets.
- **[Explore fundamental relational data concepts](https://learn.microsoft.com/en-us/training/modules/explore-relational-data-offerings/)**, Microsoft Learn. A short, free module on tables, keys, and normalization, with a quiz at the end.
- **[Database normalization description](https://learn.microsoft.com/en-us/office/troubleshoot/access/database-normalization-description)**, Microsoft. The formal name for "store each fact once" is normalization, and this is a plain-language walk through its rules, using the same kind of customer and order tables.
- **[The SQLite Query Planner](https://www.sqlite.org/queryplanner.html)**. Where the "declarative, not procedural" quote comes from, and a readable look at how a database works out the steps you didn't write.
- **Practice:** [SQLBolt](https://sqlbolt.com/) runs short interactive exercises in the browser, and its first lessons pair well with `02 - SQL Basics`.

For interview questions on this topic with model answers, see the [study guide](<study-guides/01-what-is-a-database.md>).

## What's next

`02 - SQL Basics` is where this becomes something you type. It uses a single table of seven orders, introduces `SELECT`, `WHERE`, and `ORDER BY`, and builds one query to answer one question: which completed orders were over $100?

---

[Course Overview](<00-course-overview.ipynb>) · [02 - SQL Basics →](<02-sql-basics.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](LICENSE).